# 4a. PULL transfer (DSP 2025-1, eunomia RPC)

With an agreement in hand, the Consumer asks for the data. In **PULL** mode the Provider answers
the start with a data address (endpoint and token) and the Consumer fetches the data from it. The
connectors of this notebook authenticate against the backend with the bearer token stored in the
Keystore (`setup_catalog_policies_oauth.ipynb`).

Lifecycle walked here: `REQUESTED` -> `STARTED` -> `SUSPENDED` -> `STARTED` -> `TERMINATED`, plus
two refusals: a start sent by the Consumer (role rule) and a body that is plain JSON (shape rule).

**Runs on its own.** Prerequisites: `setup_catalog_policies_oauth.ipynb` and
`contract_negotiation.ipynb`, both guarded.

## 1. Prerequisites

Two guards make the notebook runnable on its own and cheap to rerun in the same kernel:
- the catalog notebook runs unless the kernel already holds its variant (`catalog_setup == "bearer"`);
  it onboards the participants first if needed;
- the negotiation runs unless the kernel already holds an agreement from `contract_negotiation.ipynb` on the loaded
  dataset (`agreement_source == ("rpc", target_id)`).

In [ ]:
if globals().get("catalog_setup") != "bearer":
    %run ./setup_catalog_policies_oauth.ipynb
if globals().get("agreement_source") != ("rpc", target_id):
    %run ./contract_negotiation.ipynb
print(f"Transferring under agreement {agreement_id}")

### How the transfer RPCs work

As in the negotiation, each participant asks its own agent to send the next DSP message, through
`/dsp/current/transfers/rpc/setup-{request,start,suspension,completion,termination}`. Bodies are
eunomia JSON-LD (`@context` the eunomia context, `@type` `RpcTransfer<Kind>Message`) and the
answer is an `RpcTransferResult`: `request`, the peer's `response`, and `transferAgentModel`, the
caller's own process (`state`, `correlation` with both pids, messages). A request must go to the
Provider of the agreement it names.

Who may send what is fixed by the protocol (DSP 9.1): only the Consumer requests, only the
Provider starts, both may suspend, complete or terminate.

In [ ]:
def transfer_step(response, expected_state):
    """Prints an RpcTransferResult and checks the caller's process reached expected_state."""
    answer = show(response)
    process = answer["transferAgentModel"]
    assert process["state"] == expected_state, f"expected {expected_state}, got {process['state']}"
    print(f"\n==> {process['state']}")
    return process

## 2. Transfer request (Consumer -> Provider)

The Consumer names the agreement and the format of the distribution it wants (`http+pull`). The
Provider resolves the connector instance of that distribution and opens its process in `REQUESTED`.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferRequestMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+pull",
}
response = requests.post(f"{data_space_consumer}/dsp/current/transfers/rpc/setup-request", headers=json_header, json=payload)
consumer_transfer = transfer_step(response, "REQUESTED")
transfer_consumer_pid = consumer_transfer["correlation"]["consumerPid"]
transfer_provider_pid = consumer_transfer["correlation"]["providerPid"]
print(f"consumerPid: {transfer_consumer_pid}\nproviderPid: {transfer_provider_pid}")

## 3. Start (Provider -> Consumer)

The Provider's `TransferStartMessage` carries the data address the Consumer pulls from: an
endpoint of the Provider's dataplane that proxies the backend described by the PULL connector.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "STARTED")

## 4. Suspend (Provider -> Consumer)

The Provider pauses access; the data address stops answering until the transfer starts again.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferSuspensionMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"],
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-suspension", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "SUSPENDED")

## 5. Resume (Provider -> Consumer)

A suspended transfer resumes with another start message, again sent by the Provider.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "STARTED")

## 6. Role check: the Consumer cannot start

Only the Provider sends a `TransferStartMessage`. The Consumer's agent refuses the RPC before
anything goes on the wire and answers a `TransferError` naming the broken rule.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
}
response = requests.post(f"{data_space_consumer}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
answer = show(response, ok=False)
assert response.status_code == 400, "the Consumer must not be able to start a transfer"

## 7. Shape check: plain JSON is refused

The RPC bodies are read as JSON-LD. Without the eunomia `@context` and an `@type` the expanded
body declares no message type, and the agent answers 400.

In [ ]:
payload = {
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
answer = show(response, ok=False)
assert response.status_code == 400, "a body without @context and @type must be refused"

## 8. Termination (Consumer -> Provider)

Either side may terminate. `TERMINATED` is terminal: the data address is revoked.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferTerminationMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
    "code": "STOPPED",
    "reason": ["Demonstration completed"],
}
response = requests.post(f"{data_space_consumer}/dsp/current/transfers/rpc/setup-termination", headers=json_header, json=payload)
consumer_transfer = transfer_step(response, "TERMINATED")

## 9. Final state on both sides

Each agent's management API (`/api/v1/transfer-agent/transfer-processes/{id}`) shows its own record
of the process, with every message exchanged.

In [ ]:
for name, agent_url, process in [("Consumer", data_space_consumer, consumer_transfer), ("Provider", data_space_provider, provider_transfer)]:
    view = requests.get(f"{agent_url}/api/v1/transfer-agent/transfer-processes/{process['id']}", headers=json_header).json()
    print(f"{name}: {view['state']}")
print()
print(json.dumps(view, indent=2))